# Channel Performance Analysis

In this notebook I explore, clean, and prepare the raw marketing data for a home decor e-commerce brand. The data covers five channels: Google Ads, Meta Ads, Influencer, Email, and Earned Media - over roughly a year.

My goal here is to get the dataset into a trustworthy, analysis-ready state before I move into Power BI for the modeling, DAX measures, and dashboard building. I run an EDA (exploratory data analysis) to understand the shape and quality of the data, fix inconsistencies I find along the way (channel naming, duplicate rows, missing revenue values), and export a clean CSV at the end.

## 1. Loading the Data

I start by importing pandas and reading the raw CSV export into a dataframe, then take a quick look at the first rows to confirm it loaded correctly.

In [28]:
# I import pandas, the core library I'll use for all data manipulation in this notebook
import pandas as pd

# I load the raw marketing data CSV into a dataframe
df = pd.read_csv(r"G:\Il mio Drive\FORMAZIONE\Data Analysis\Case Study Portfolio\Case Studies\Marketing Case Studies\Channel_performance_analysis\nordik_home_marketing_data.csv")

# I check the first few rows to make sure the data loaded as expected
df.head()

,date,channel,campaign,spend,impressions,clicks,conversions,revenue
0,2026-06-03,Google Ads,Shopping - Catalog,207.67,4652,104,0,0.00
1,2026-03-09,Influencer,Micro-influencers Home&Deco,81.88,18453,809,7,419.83
2,2026-02-12,Google Ads,Search - Generic,294.22,18860,319,11,762.50
3,2026-01-10,meta ads,Prospecting - Broad,249.22,8599,112,1,52.92
4,2026-04-21,Email,Flow - Abandoned Cart,0.00,11973,288,15,999.44


## 2. Exploratory Data Analysis (EDA)

Before touching anything, I want to understand what I'm working with: how many rows and columns, what data types pandas inferred, and whether there are any obvious data quality issues.

### 2.1 Shape, structure, and summary statistics

In [29]:
# I check the number of rows and columns in the dataset
df.shape

(4760, 8)

In [30]:
# I look at column names, data types, and non-null counts to spot any structural issues early
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4760 entries, 0 to 4759
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         4760 non-null   object 
 1   channel      4760 non-null   object 
 2   campaign     4760 non-null   object 
 3   spend        4760 non-null   float64
 4   impressions  4760 non-null   int64  
 5   clicks       4760 non-null   int64  
 6   conversions  4760 non-null   int64  
 7   revenue      4720 non-null   float64
dtypes: float64(2), int64(3), object(3)
memory usage: 297.6+ KB


In [31]:
# I get summary statistics for the numeric columns, to sanity-check ranges and spot outliers
df.describe()

,spend,impressions,clicks,conversions,revenue
count,4760.000000,4760.000000,4760.000000,4760.000000,4720.000000
mean,119.382143,10589.277731,281.796849,8.589286,494.149023
std,117.538905,5561.141802,193.902178,8.630274,524.523157
min,0.000000,866.000000,12.000000,0.000000,0.000000
25%,0.000000,6466.750000,139.000000,2.000000,125.970000
50%,112.010000,9651.500000,233.500000,6.000000,326.890000
75%,205.317500,13605.000000,380.250000,12.000000,684.377500
max,566.510000,31960.000000,1398.000000,65.000000,4972.170000


### 2.2 Checking for missing values

In [32]:
# I count how many rows have a missing 'revenue' value — this is the only column with nulls I've noticed so far
df["revenue"].isnull().sum()

np.int64(40)

### 2.3 Checking unique values per column

In [33]:
# I check how many unique values each column has, to get a sense of the data's granularity (e.g. how many distinct channels, campaigns, dates)
df.nunique()

date            365
channel           6
campaign         13
spend          2792
impressions    4148
clicks          792
conversions      53
revenue        4206
dtype: int64

In [34]:
# I look at the distribution of rows across dates, to confirm the data is spread reasonably evenly over the period
df["date"].value_counts()

date
2026-02-14    14
2025-09-13    14
2026-02-01    14
2026-08-12    14
2025-10-14    14
              ..
2026-03-23    13
2026-07-11    13
2026-07-24    13
2026-01-29    13
2026-02-12    13
Name: count, Length: 365, dtype: int64

In [35]:
# I check the distinct values in 'channel' — this is where I first spot the naming inconsistency (see next section)
df["channel"].value_counts()

channel
Email           1101
Google Ads      1096
Meta Ads        1065
Influencer       733
Earned Media     732
meta ads          33
Name: count, dtype: int64

### 2.4 Fixing inconsistent channel naming

When I checked the channel values above, I noticed 'meta ads' (lowercase) sitting alongside 'Meta Ads' — the same channel being counted as two different categories. I standardize this before doing any further analysis, since it would otherwise silently split my Meta Ads numbers in two.

In [36]:
# I strip any leading/trailing whitespace from channel names, in case that's contributing to inconsistencies
df["channel"] = df["channel"].str.strip()

In [37]:
# I replace the lowercase 'meta ads' variant with the correctly capitalized 'Meta Ads', so it's no longer treated as a separate category
df["channel"] = df["channel"].replace("meta ads", "Meta Ads")
# alternative approach I considered: df["channel"] = df["channel"].str.title()


In [38]:
# I re-check the channel values to confirm the fix worked — I should now see only 5 clean categories
df["channel"].value_counts()

channel
Email           1101
Meta Ads        1098
Google Ads      1096
Influencer       733
Earned Media     732
Name: count, dtype: int64

## 2 (continued EDA) Exploring Campaigns

In [39]:
# I check the distinct campaign names and how many rows each has
df["campaign"].value_counts()

campaign
Micro-influencers Home&Deco      367
Flow - Abandoned Cart            367
Newsletter Weekly                367
Flow - Post Purchase             367
Organic Social Mentions          367
Retargeting - Cart Abandoners    366
Prospecting - Broad              366
Search - Generic                 366
Lookalike - Top Customers        366
Macro Campaign Q4                366
Shopping - Catalog               365
Search - Brand                   365
Press Mentions                   365
Name: count, dtype: int64

In [40]:
# I group by channel to see which campaigns belong to each channel — this helps me understand the structure of the data
df.groupby("channel").agg(campaign=("campaign","unique"))

,campaign
channel,
Earned Media,"[Organic Social Mentions, Press Mentions]"
Email,"[Flow - Abandoned Cart, Flow - Post Purchase, ..."
Google Ads,"[Shopping - Catalog, Search - Generic, Search ..."
Influencer,"[Micro-influencers Home&Deco, Macro Campaign Q4]"
Meta Ads,"[Prospecting - Broad, Lookalike - Top Customer..."


In [41]:
# same check, written a slightly different way, just to confirm the grouping is consistent
df.groupby("channel")["campaign"].unique()

channel
Earned Media            [Organic Social Mentions, Press Mentions]
Email           [Flow - Abandoned Cart, Flow - Post Purchase, ...
Google Ads      [Shopping - Catalog, Search - Generic, Search ...
Influencer       [Micro-influencers Home&Deco, Macro Campaign Q4]
Meta Ads        [Prospecting - Broad, Lookalike - Top Customer...
Name: campaign, dtype: object

In [42]:
# I list every unique channel-campaign combination, sorted alphabetically, as a final visual check that everything looks right
df[["channel","campaign"]].drop_duplicates().sort_values(["channel","campaign"],ascending=[True,True])

,channel,campaign
5,Earned Media,Organic Social Mentions
13,Earned Media,Press Mentions
4,Email,Flow - Abandoned Cart
7,Email,Flow - Post Purchase
11,Email,Newsletter Weekly
28,Google Ads,Search - Brand
2,Google Ads,Search - Generic
0,Google Ads,Shopping - Catalog
30,Influencer,Macro Campaign Q4
1,Influencer,Micro-influencers Home&Deco


### 2.5 Checking for duplicate rows

In [43]:
# I count how many fully duplicated rows exist in the dataset
df.duplicated().sum()

np.int64(13)

In [44]:
# I drop the duplicate rows I found, keeping only the first occurrence of each
df = df.drop_duplicates()

In [45]:
# I confirm there are no duplicates left after the cleanup
df.duplicated().sum()

np.int64(0)

### 2.6 Handling missing revenue values

Earlier I found 40 rows with a missing 'revenue' value. Instead of dropping them or filling them all with the same value, I want to understand *why* they're missing before deciding how to handle them.

In [46]:
# I look at the rows with missing revenue, sorted by conversions descending, to see if there's a pattern (e.g. do they have conversions or not?)
df[df["revenue"].isnull()].sort_values(by="conversions",ascending=False)

,date,channel,campaign,spend,impressions,clicks,conversions,revenue
3581,2026-02-01,Google Ads,Search - Brand,230.64,11692,495,31,NaN
75,2025-11-27,Google Ads,Search - Brand,305.06,18180,530,23,NaN
2077,2026-02-09,Google Ads,Search - Generic,147.13,17336,599,23,NaN
1944,2025-09-27,Google Ads,Search - Brand,350.15,13231,390,22,NaN
4410,2025-11-23,Meta Ads,Retargeting - Cart Abandoners,366.49,24412,427,20,NaN
2402,2025-11-30,Email,Flow - Abandoned Cart,0.00,6492,239,18,NaN
3489,2025-12-13,Email,Flow - Abandoned Cart,0.00,7039,187,16,NaN
2183,2026-05-27,Meta Ads,Lookalike - Top Customers,201.16,11786,482,16,NaN
4103,2025-12-31,Email,Flow - Post Purchase,0.00,5654,182,14,NaN
3628,2025-09-10,Google Ads,Shopping - Catalog,245.68,12703,268,14,NaN


In [47]:
# I group by channel and campaign to count, for each group, how many rows there are in total and how many have a null revenue — this helps me see whether the nulls are spread across many campaigns or concentrated in a few
df.groupby(["channel","campaign"]).agg(rows=("revenue","size"), nulls=("revenue", lambda x: x.isnull().sum())).reset_index()

,channel,campaign,rows,nulls
0,Earned Media,Organic Social Mentions,366,3
1,Earned Media,Press Mentions,365,6
2,Email,Flow - Abandoned Cart,365,2
3,Email,Flow - Post Purchase,365,1
4,Email,Newsletter Weekly,365,4
5,Google Ads,Search - Brand,365,6
6,Google Ads,Search - Generic,365,3
7,Google Ads,Shopping - Catalog,365,3
8,Influencer,Macro Campaign Q4,365,2
9,Influencer,Micro-influencers Home&Deco,366,3


In [48]:
# Case 1: where revenue is missing AND conversions is 0, it makes sense to fill revenue with 0 — no conversions means no revenue was generated
df.loc[(df["revenue"].isnull()) & (df["conversions"]==0), "revenue"] = 0

In [49]:
# Case 2: where revenue is missing BUT there were conversions, filling with 0 would understate performance — instead, I estimate the missing value using the average revenue for that same channel-campaign combination
df.loc[
    (df["revenue"].isnull())
    &
    (df["conversions"]!=0),
    "revenue"
] = df.groupby(["channel","campaign"])["revenue"].transform("mean")

In [50]:
# I confirm there are no missing revenue values left after applying both fixes
df["revenue"].isnull().sum()

np.int64(0)

## 3. Data Type Conversion

The 'date' column was read in as a generic object/string type. I convert it to a proper datetime type, which I'll need for any time-based analysis later (trends, period-over-period comparisons, etc.) in Power BI.

In [51]:
# I check the current dtype of the columns, and preview the first 3 rows, before converting the date column
# datatype conversion
df.info()

df.head(3)

<class 'pandas.core.frame.DataFrame'>
Index: 4747 entries, 0 to 4759
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         4747 non-null   object 
 1   channel      4747 non-null   object 
 2   campaign     4747 non-null   object 
 3   spend        4747 non-null   float64
 4   impressions  4747 non-null   int64  
 5   clicks       4747 non-null   int64  
 6   conversions  4747 non-null   int64  
 7   revenue      4747 non-null   float64
dtypes: float64(2), int64(3), object(3)
memory usage: 462.8+ KB


,date,channel,campaign,spend,impressions,clicks,conversions,revenue
0,2026-06-03,Google Ads,Shopping - Catalog,207.67,4652,104,0,0.00
1,2026-03-09,Influencer,Micro-influencers Home&Deco,81.88,18453,809,7,419.83
2,2026-02-12,Google Ads,Search - Generic,294.22,18860,319,11,762.50


In [52]:
# I convert the 'date' column to a proper datetime type; errors='coerce' means any value that can't be parsed becomes NaT instead of raising an error
df["date"]=pd.to_datetime(df["date"],errors="coerce")

# I check the result to confirm the dtype changed correctly
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 4747 entries, 0 to 4759
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         4747 non-null   datetime64[ns]
 1   channel      4747 non-null   object        
 2   campaign     4747 non-null   object        
 3   spend        4747 non-null   float64       
 4   impressions  4747 non-null   int64         
 5   clicks       4747 non-null   int64         
 6   conversions  4747 non-null   int64         
 7   revenue      4747 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(2)
memory usage: 462.8+ KB


,date,channel,campaign,spend,impressions,clicks,conversions,revenue
0,2026-06-03,Google Ads,Shopping - Catalog,207.67,4652,104,0,0.00
1,2026-03-09,Influencer,Micro-influencers Home&Deco,81.88,18453,809,7,419.83
2,2026-02-12,Google Ads,Search - Generic,294.22,18860,319,11,762.50
3,2026-01-10,Meta Ads,Prospecting - Broad,249.22,8599,112,1,52.92
4,2026-04-21,Email,Flow - Abandoned Cart,0.00,11973,288,15,999.44


In [53]:
# I double-check that the conversion didn't introduce any new missing dates (which would happen if coerce had to fall back to NaT anywhere)
df["date"].isnull().sum()

np.int64(0)

## 4. Wrapping Up: Exporting the Clean Dataset

At this point the dataframe is clean: channel names are standardized, duplicates are removed, missing revenue values are handled with clear logic, and the date column has the correct type. I export it to a fresh CSV, which I then load into Power BI to continue the analysis: building the data model, writing the DAX measures, and designing the dashboard.

In [54]:
# I export the cleaned dataframe to a new CSV file, ready to be loaded into Power BI
df.to_csv("channel_data.csv", index=False)